In [1]:
import polars as pl
import polars_corpus as plc

In [2]:
bnc = pl.read_parquet('bnc.parquet')

In [3]:
bnc.schema

Schema([('token', String),
        ('lemma', String),
        ('pos', String),
        ('c5', String),
        ('sent_tag', String),
        ('mode', Categorical),
        ('text_type', Categorical),
        ('file_id', Categorical),
        ('speaker_id', String)])

In [4]:
matches = plc.search(bnc, '[token="small"] [pos="ADJ"]* [pos="SUBST"]').head(30)

In [8]:
matches.view(window=10)

In [6]:
matches.view(chunk_tag='sent_tag')

In [35]:
itables.show(sent.with_columns(pl.all().list.join(' ')),
             classes="compact",
             columnDefs=[
                 {"className": "dt-right", "targets": [0]},
                 {"className": "dt-center", "targets": [1]},
                 {"className": "dt-left", "targets": [2]}],)

Loading ITables v2.5.2 from the internet... (need help?)


In [48]:
itables.show(sent.select(pl.concat_list(pl.all()).list.join(" ").alias('sentence')),
             classes="compact",
             columnDefs=[
                 {"className": "dt-left", "targets": [0]}], )


Loading ITables v2.5.2 from the internet... (need help?)


In [49]:
def esc(e: pl.Expr) -> pl.Expr:
    # escape once per row-string
    return (
        e.str.replace_all("&", "&amp;")
        .str.replace_all("<", "&lt;")
        .str.replace_all(">", "&gt;")
    )

sentence_expr = pl.concat_str(
    [
        # Left context (optional trailing space)
        esc(pl.col("token_left_context").list.join(" ", ignore_nulls=True).fill_null(""))
        .map_elements(lambda s: s + (" " if s else ""), return_dtype=pl.Utf8),

        # Bolded token (already escaped)
        pl.format(
            "<b>{}</b>",
            esc(pl.col("token").list.join(" ", ignore_nulls=True).fill_null(""))
        ),

        # Right context (optional leading space)
        esc(pl.col("token_right_context").list.join(" ", ignore_nulls=True).fill_null(""))
        .map_elements(lambda s: (" " + s) if s else "", return_dtype=pl.Utf8),
    ],
    separator=""
).alias("sentence")

out = sent.select(sentence_expr)

itables.show(
    out,
    allow_html=["sentence"],  # safe: only our <b> tags remain; everything else is escaped
    classes="compact",
    columnDefs=[{"className": "dt-left", "targets": [0]}],
)

out = sent.select(
    pl.concat_str(
        [add_trailing_space(L), pl.format("<b>{}</b>", T), add_leading_space(R)],
        separator=""
    ).alias("sentence")
)



Schema([('token_left_context', List(String)),
        ('token', List(String)),
        ('token_right_context', List(String))])